# Prepare a ZincSight batch for Tamarind

Enter many IDs and/or upload structure files to prepare a Tamarind batch. **Predictions run on Tamarind**; this notebook downloads and packages inputs only. Use a standard CPU Colab runtime. No API key, account connection, or GPU is needed.

1. Run **Enter inputs**, then paste IDs separated by commas, spaces, tabs, or newlines and choose any upload options.
2. Run **Prepare ZIP** and select any requested ID lists (`.txt`, `.csv`, `.tsv`) or structures/archives.
3. Run **Download**. Upload **every generated ZIP part together** under **Protein structures or batch archives** on the ZincSight Tamarind page. Allow multiple browser downloads if prompted.

There is no fixed structure-count or protein-size cap. The preparer uses temporary disk and splits by Tamarind's **200 MB per-file upload transport limit** when needed, preserving every input and its provenance manifest. Actual prediction capacity depends on the Tamarind job's CPU, memory, disk, and time allocation.

TXT lists may contain quoted IDs; CSV/TSV tables recognize ID headers including `structure_id`, `model_id`, `alphafold_model_id`, `uniprot_accession` and UniProt `Entry`. The first recognized ID column is used. Known `.pdb`/`.cif`/`.mmcif`/`.ent` filename suffixes (optionally `.gz`) are accepted as ID spellings. Explicit AlphaFold fragments and versions are preserved. Missing selected ID cells and malformed lists produce an error before downloads.

IDs may be PDB/assembly, UniProt, complete AlphaFold model, MGYP, or complete TED domain IDs. Examples: `1KLS`, `2A0S-assembly1`, `P12345`, `AF-P12345-F2-model_v4`, `AF-A0A287EUI1-F1-model_v4_TED01`. Uploads may be PDB/mmCIF/ENT (also gzip compressed), ZIP, TAR, TAR.GZ, or TGZ.

The standalone preparer is embedded below; no remote code is fetched or executed. See the [integration README](https://github.com/MECHTI1/ZincSight/blob/feature/tamarind-integration/integrations/tamarind/README.md) and [ZincSight paper](https://doi.org/10.1002/pro.70350).


In [ ]:
#@title 1. Enter inputs { display-mode: "form" }
import hashlib
from pathlib import Path
import types
import ipywidgets as widgets
from IPython.display import display

# Exact standalone preparer source, bundled so no remote code is executed.
PREPARER_SOURCE = '#!/usr/bin/env python3\n"""Prepare Colab-style inputs before uploading them to offline Tamarind jobs.\n\nThis companion CLI runs on the user\'s networked computer, not inside Tamarind.\nIt accepts public structure IDs, ID-list files, structures, folders, and archives,\nthen produces upload-sized ZIP parts and a provenance manifest. It never reads an API\nkey, submits a job, or includes credentials or absolute local paths in outputs.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport gzip\nimport hashlib\nimport io\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport re\nimport stat\nimport tarfile\nimport tempfile\nimport time\nimport urllib.error\nimport urllib.parse\nimport urllib.request\nimport zipfile\nimport zlib\n\nUPLOAD_MAX_BYTES = 200_000_000  # Tamarind\'s per-file upload transport limit.\nPROVIDER_PREFIXES = {\n    "files.rcsb.org": "/download/",\n    "alphafold.ebi.ac.uk": "/files/",\n    "api.esmatlas.com": "/fetchPredictedStructure/",\n    "ted.cathdb.info": "/api/v1/files/",\n}\nUNIPROT = r"(?:[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9](?:[A-Z][A-Z0-9]{2}[0-9]){1,2})"\nAF_MODEL = rf"AF-({UNIPROT})-F([1-9][0-9]*)-model_v([1-9][0-9]*)"\nID_HEADERS = {"id", "ids", "identifier", "identifiers", "structure_id", "structure_ids",\n              "query_id", "pdb", "pdb_id", "uniprot", "uniprot_id", "accession",\n              "entry", "uniprot_accession", "model_id", "alphafold_model_id"}\n\n\ndef _id_text(text: str) -> str:\n    if not isinstance(text, str) or re.search(r"[\\x00-\\x08\\x0b\\x0c\\x0e-\\x1f\\x7f]", text):\n        raise ValueError("ID lists must contain text without binary control characters.")\n    return text.strip().lstrip("\\ufeff").strip()\n\n\ndef _unquote_id(value: str) -> str:\n    value = _id_text(value)\n    if value.startswith((\'"\', "\'")):\n        if len(value) < 2 or value[-1] != value[0]:\n            raise ValueError("An identifier has unmatched quotes.")\n        value = value[1:-1].strip().lstrip("\\ufeff").strip()\n    if \'"\' in value or "\'" in value:\n        raise ValueError("An identifier has malformed quoting.")\n    return value\n\n\ndef parse_ids(text: str) -> list[str]:\n    """Accept quoted/BOM-prefixed comma/whitespace IDs and optional # comments."""\n    clean = "\\n".join(line.split("#", 1)[0] for line in _id_text(text).splitlines())\n    tokens, previous_end = [], None\n    for match in re.finditer(r\'\'\'(?:"[^"]*"|\'[^\']*\'|[^\\s,]+)\'\'\', clean):\n        if previous_end is not None and not clean[previous_end:match.start()]:\n            raise ValueError("Quoted identifiers must be separated by commas or whitespace.")\n        value = _unquote_id(match.group())\n        tokens.extend(part for part in re.split(r"[\\s,]+", value) if part)\n        previous_end = match.end()\n    return list(dict.fromkeys(tokens))\n\n\ndef read_id_list(data: bytes, suffix: str = ".txt", *, max_bytes=None) -> list[str]:\n    if not data or (max_bytes is not None and len(data) > max_bytes):\n        raise ValueError("ID-list files must be nonempty and respect any configured byte guard.")\n    text = _id_text(data.decode("utf-8-sig"))\n    # Detect the table from its header, not only its filename: spreadsheets are\n    # often saved as .txt or given the wrong .csv/.tsv extension. Keep the first\n    # recognized ID column; metadata columns never become extra identifiers.\n    delimiters = ("\\t", ",") if suffix.lower() == ".tsv" else (",", "\\t")\n    for delimiter in delimiters:\n        rows = (row for row in csv.reader(io.StringIO(text), delimiter=delimiter,\n                                           skipinitialspace=True, strict=True)\n                if row and any(value.strip() for value in row)\n                and not row[0].lstrip().startswith("#"))\n        try:\n            first = next(rows, None)\n        except csv.Error:\n            continue\n        if first is None:\n            continue\n        header = [re.sub(r"\\s+", "_", value.strip().lower()) for value in first]\n        index = next((i for i, value in enumerate(header) if value in ID_HEADERS), None)\n        if index is not None:\n            try:\n                values = []\n                for row in rows:\n                    selected = parse_ids(row[index]) if len(row) > index else []\n                    if not selected:\n                        raise ValueError(f"A table row is missing a value for the selected ID column: {header[index]}.")\n                    values.extend(selected)\n            except csv.Error as exc:\n                raise ValueError("The ID-list table has malformed quoting.") from exc\n            return list(dict.fromkeys(values))\n    return parse_ids(text)\n\n\ndef resolve_identifier(identifier: str) -> dict:\n    """Resolve identifiers only through fixed public-provider URL templates."""\n    value = _unquote_id(identifier)\n    value = re.sub(r"\\.(?:pdb|cif|mmcif|ent)(?:\\.gz)?$", "", value, flags=re.I)\n    pdb = re.fullmatch(r"([0-9][A-Z0-9]{3})(?:-assembly([1-9][0-9]{0,2}))?", value, re.I)\n    if pdb:\n        canonical = pdb[1].upper() + (f"-assembly{pdb[2]}" if pdb[2] else "")\n        return {"canonical_id": canonical, "source_url": f"https://files.rcsb.org/download/{canonical}.cif",\n                "file_format": "cif", "filename": canonical + ".cif"}\n    if re.fullmatch(r"MGYP[0-9]{12}", value, re.I):\n        canonical = value.upper()\n        return {"canonical_id": canonical,\n                "source_url": f"https://api.esmatlas.com/fetchPredictedStructure/{canonical}.pdb",\n                "file_format": "pdb", "filename": canonical + ".pdb"}\n    ted = re.fullmatch(AF_MODEL + r"_TED([0-9]{2,4})", value, re.I)\n    if ted and int(ted[4]) > 0:\n        canonical = f"AF-{ted[1].upper()}-F{ted[2]}-model_v{ted[3]}_TED{ted[4]}"\n        return {"canonical_id": canonical, "source_url": f"https://ted.cathdb.info/api/v1/files/{canonical}.pdb",\n                "file_format": "pdb", "filename": canonical + ".pdb"}\n    model = re.fullmatch(AF_MODEL, value, re.I)\n    if model:\n        canonical = f"AF-{model[1].upper()}-F{model[2]}-model_v{model[3]}"\n    elif re.fullmatch(UNIPROT, value, re.I):\n        canonical = f"AF-{value.upper()}-F1-model_v6"\n    else:\n        raise ValueError("Unsupported identifier; use a PDB/assembly, UniProt, full AlphaFold, MGYP or TED domain ID.")\n    return {"canonical_id": canonical, "source_url": f"https://alphafold.ebi.ac.uk/files/{canonical}.cif",\n            "file_format": "cif", "filename": canonical + ".cif"}\n\n\ndef validate_provider_url(url: str) -> None:\n    parsed = urllib.parse.urlsplit(url)\n    prefix = PROVIDER_PREFIXES.get(parsed.hostname or "")\n    if (parsed.scheme != "https" or prefix is None or not parsed.path.startswith(prefix)\n            or parsed.username is not None or parsed.password is not None\n            or parsed.port not in (None, 443) or parsed.query or parsed.fragment):\n        raise ValueError("Only fixed HTTPS structure-provider URLs are permitted.")\n\n\nclass ProviderRedirectHandler(urllib.request.HTTPRedirectHandler):\n    def redirect_request(self, request, fp, code, msg, headers, newurl):\n        validate_provider_url(newurl)\n        return super().redirect_request(request, fp, code, msg, headers, newurl)\n\n\ndef read_bounded(stream, limit: int | None, label: str) -> bytes:\n    data = stream.read() if limit is None else stream.read(limit + 1)\n    if limit is not None and len(data) > limit:\n        raise ValueError(f"{label} exceeds its allowed byte limit.")\n    return data\n\n\ndef download_structure(url: str, *, max_bytes: int | None = None, attempts: int = 3,\n                       timeout: int = 30, opener=None, sleep=time.sleep) -> bytes:\n    validate_provider_url(url)\n    if not 1 <= attempts <= 3 or not 1 <= timeout <= 60:\n        raise ValueError("Downloads allow 1–3 attempts and a 1–60 second timeout.")\n    opener = opener or urllib.request.build_opener(ProviderRedirectHandler())\n    for attempt in range(attempts):\n        try:\n            request = urllib.request.Request(url, headers={"User-Agent": "ZincSight-Tamarind-Input-Preparer/1.0"})\n            with opener.open(request, timeout=timeout) as response:\n                validate_provider_url(response.geturl())\n                length = response.headers.get("Content-Length")\n                if length is not None and max_bytes is not None and int(length) > max_bytes:\n                    raise ValueError("Downloaded structure exceeds the allowed byte limit.")\n                if "text/html" in response.headers.get("Content-Type", "").lower():\n                    raise ValueError("The provider returned an HTML page instead of a structure.")\n                return read_bounded(response, max_bytes, "Downloaded structure")\n        except urllib.error.HTTPError as exc:\n            if exc.code not in (429, 500, 502, 503, 504) or attempt + 1 == attempts:\n                raise ValueError(f"Public structure provider returned HTTP {exc.code}.") from exc\n        except (urllib.error.URLError, TimeoutError) as exc:\n            if attempt + 1 == attempts:\n                raise ValueError("Public structure download failed after bounded retries.") from exc\n        sleep(0.25 * (2 ** attempt))\n    raise AssertionError("Unreachable download retry state")\n\n\ndef structure_format(name: str) -> str | None:\n    lower = name.lower()\n    if lower.endswith(".gz"):\n        lower = lower[:-3]\n    suffix = PurePosixPath(lower).suffix\n    return "pdb" if suffix in (".pdb", ".ent") else "cif" if suffix in (".cif", ".mmcif") else None\n\n\ndef structure_bytes(data: bytes, name: str, limit: int | None) -> bytes:\n    if name.lower().endswith(".gz"):\n        with gzip.GzipFile(fileobj=io.BytesIO(data)) as compressed:\n            data = read_bounded(compressed, limit, "Decompressed structure")\n    if not data or (limit is not None and len(data) > limit):\n        raise ValueError("Each structure must be nonempty and respect any configured byte guard.")\n    text = data.decode("utf-8-sig")\n    if "\\x00" in text:\n        raise ValueError("Structures must be text PDB or mmCIF files.")\n    fmt = structure_format(name)\n    if fmt == "pdb" and not any(line.startswith("ATOM  ") for line in text.splitlines()):\n        raise ValueError(f"{PurePosixPath(name).name}: PDB input must contain protein ATOM records.")\n    if fmt == "cif" and "_atom_site." not in text:\n        raise ValueError(f"{PurePosixPath(name).name}: mmCIF input must contain an atom_site table.")\n    return data\n\n\ndef safe_archive_name(name: str) -> str:\n    clean = name.rstrip("/")\n    # Standard tar tools commonly emit ./member paths; normalize that prefix\n    # before detecting traversal and collisions, without allowing internal dots.\n    while clean.startswith("./"):\n        clean = clean[2:]\n    if (not clean or "\\\\" in clean or clean.startswith("/") or re.match(r"^[A-Za-z]:", clean)\n            or any(part in ("", ".", "..") for part in clean.split("/"))):\n        raise ValueError("Archive paths must be relative and cannot contain traversal components.")\n    return PurePosixPath(clean).as_posix()\n\n\ndef archive_structures(path: Path, *, max_file_bytes=None, max_total_bytes=None, max_entries=None):\n    seen, count, total = set(), 0, 0\n\n    def admit(name: str, size: int):\n        nonlocal count, total\n        name = safe_archive_name(name)\n        key = name.casefold()\n        if key in seen:\n            raise ValueError("Archive contains duplicate or case-colliding paths.")\n        seen.add(key)\n        count += 1\n        total += size\n        if ((max_entries is not None and count > max_entries)\n                or (max_total_bytes is not None and total > max_total_bytes)):\n            raise ValueError("Archive exceeds its entry-count or uncompressed-byte limit.")\n        return name\n\n    if path.suffix.lower() == ".zip":\n        with zipfile.ZipFile(path) as archive:\n            for info in archive.infolist():\n                if info.is_dir() and info.filename in (".", "./"):\n                    continue\n                name = admit(info.filename, info.file_size)\n                kind = stat.S_IFMT(info.external_attr >> 16)\n                if kind not in (0, stat.S_IFREG, stat.S_IFDIR) or info.flag_bits & 1:\n                    raise ValueError("Archives cannot contain links, special files or encrypted entries.")\n                if info.is_dir():\n                    continue\n                if structure_format(name):\n                    if max_file_bytes is not None and info.file_size > max_file_bytes:\n                        raise ValueError("An archived structure exceeds the allowed byte limit.")\n                    with archive.open(info) as stream:\n                        yield name, read_bounded(stream, max_file_bytes, "Archived structure")\n    else:\n        with tarfile.open(path, "r|*") as archive:\n            for info in archive:\n                if info.isdir() and info.name in (".", "./"):\n                    continue\n                name = admit(info.name, info.size)\n                if not (info.isfile() or info.isdir()):\n                    raise ValueError("Archives cannot contain links or special files.")\n                if info.isfile() and structure_format(name):\n                    if max_file_bytes is not None and info.size > max_file_bytes:\n                        raise ValueError("An archived structure exceeds the allowed byte limit.")\n                    with archive.extractfile(info) as stream:\n                        yield name, read_bounded(stream, max_file_bytes, "Archived structure")\n\n\ndef is_archive(path: Path) -> bool:\n    return path.name.lower().endswith((".zip", ".tar", ".tar.gz", ".tgz"))\n\n\ndef _guard(value, label):\n    if value is not None and (isinstance(value, bool) or not isinstance(value, int) or value < 1):\n        raise ValueError(f"{label} must be a positive integer when supplied.")\n\n\ndef _manifest(entries, identifiers):\n    return {"schema_version": 1, "prepared_by": "ZincSight Tamarind input preparer",\n            "requires_network_at_runtime": False, "requested_ids": identifiers,\n            "structure_count": len(entries),\n            "total_structure_bytes": sum(entry["size_bytes"] for entry in entries),\n            "structures": entries}\n\n\ndef _json_bytes(value):\n    return (json.dumps(value, indent=2, ensure_ascii=True, allow_nan=False) + "\\n").encode()\n\n\ndef _compressed_size(path):\n    compressor, size = zlib.compressobj(level=6, wbits=-15), 0\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            size += len(compressor.compress(block))\n    return size + len(compressor.flush())\n\n\ndef _write_archive(path, manifest, blobs):\n    with zipfile.ZipFile(path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:\n        for name, blob in [("manifest.json", _json_bytes(manifest)),\n                           *((entry["path"], blobs[entry["path"]]) for entry in manifest["structures"])]:\n            info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))\n            info.compress_type, info.external_attr = zipfile.ZIP_DEFLATED, 0o100644 << 16\n            with archive.open(info, "w") as target:\n                if isinstance(blob, bytes):\n                    target.write(blob)\n                else:\n                    with blob.open("rb") as stream:\n                        for block in iter(lambda: stream.read(1024 * 1024), b""):\n                            target.write(block)\n\n\ndef prepare_inputs(output_zip: str | Path, *, ids: str = "", id_files=(), sources=(),\n                   downloader=download_structure, max_structures=None, max_file_bytes=None,\n                   max_total_bytes=None, max_archive_entries=None, max_folder_entries=None,\n                   max_id_file_bytes=None, upload_max_bytes=UPLOAD_MAX_BYTES,\n                   overwrite: bool = False) -> dict:\n    """Prepare every input, splitting only by the upload transport\'s byte limit.\n\n    Count, file, total, and archive/folder guards are optional user choices.\n    Structure content is spooled to disk rather than retained for the whole batch.\n    Every returned archive has its own valid, hash-bearing subset manifest.\n    """\n    for label, value in (("Structure-count guard", max_structures), ("Structure-byte guard", max_file_bytes),\n                         ("Total-byte guard", max_total_bytes), ("Archive-entry guard", max_archive_entries),\n                         ("Folder-entry guard", max_folder_entries), ("ID-list byte guard", max_id_file_bytes),\n                         ("Upload-byte limit", upload_max_bytes)):\n        _guard(value, label)\n    if upload_max_bytes is None or upload_max_bytes > UPLOAD_MAX_BYTES:\n        raise ValueError("Tamarind upload parts cannot exceed 200 MB.")\n    output = Path(output_zip).resolve()\n    manifest_path = output.with_suffix(".manifest.json")\n    if output.suffix.lower() != ".zip":\n        raise ValueError("The prepared output must use a .zip extension.")\n    if not overwrite and (output.exists() or manifest_path.exists()):\n        raise FileExistsError("An output already exists; choose another path or use --force.")\n    if max_id_file_bytes is not None and len(ids.encode()) > max_id_file_bytes:\n        raise ValueError("Identifier text exceeds the configured byte guard.")\n    identifiers = parse_ids(ids)\n    entries, blobs, local_entries, total_bytes = [], {}, [], 0\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory(prefix="zincsight-preparation-") as scratch:\n        work = Path(scratch)\n\n        def read_ids(path):\n            if path.is_symlink() or not path.is_file():\n                raise ValueError("ID-list inputs must be regular files, not links.")\n            with path.open("rb") as stream:\n                return read_id_list(read_bounded(stream, max_id_file_bytes, "ID list"), path.suffix,\n                                    max_bytes=max_id_file_bytes)\n\n        def collect(name, data, metadata):\n            if max_structures is not None and len(local_entries) >= max_structures:\n                raise ValueError("Local inputs exceed the configured structure-count guard.")\n            raw_path = work / f"local-{len(local_entries)}"\n            raw_path.write_bytes(data)\n            local_entries.append((name, raw_path, metadata))\n\n        def is_own_output(path):\n            path = path.resolve()\n            return (path in (output, manifest_path) or path.is_relative_to(work)\n                    or (path.parent == output.parent\n                        and re.fullmatch(re.escape(output.stem) + r"\\.part[0-9]+\\.zip", path.name)))\n\n        for filename in id_files:\n            identifiers.extend(read_ids(Path(filename)))\n        files_seen = 0\n        for source in sources:\n            path = Path(source)\n            if path.is_symlink() or not path.exists():\n                raise ValueError("Local inputs must exist and cannot be symbolic links.")\n            if path.is_dir():\n                paths = []\n                for item in path.rglob("*"):\n                    if is_own_output(item):\n                        continue\n                    files_seen += 1\n                    if max_folder_entries is not None and files_seen > max_folder_entries:\n                        raise ValueError("Input folders exceed the configured entry-count guard.")\n                    paths.append(item)\n                paths.sort()\n            else:\n                paths = [path]\n            for item in paths:\n                if item.is_symlink():\n                    raise ValueError("Input folders cannot contain symbolic links.")\n                if not item.is_file() or is_own_output(item):\n                    continue\n                if item.suffix.lower() in (".txt", ".csv", ".tsv") and not path.is_dir():\n                    identifiers.extend(read_ids(item))\n                elif is_archive(item):\n                    for name, data in archive_structures(item, max_file_bytes=max_file_bytes,\n                                                         max_total_bytes=max_total_bytes, max_entries=max_archive_entries):\n                        collect(name, data, {"source_type": "archive", "archive_name": item.name})\n                elif structure_format(item.name):\n                    with item.open("rb") as stream:\n                        collect(item.name, read_bounded(stream, max_file_bytes, "Structure"), {"source_type": "local_file"})\n                elif not path.is_dir():\n                    raise ValueError(f"Unsupported input file type: {item.name}")\n        identifiers = list(dict.fromkeys(identifiers))\n        resolved, seen_urls = [], set()\n        for identifier in identifiers:\n            metadata = resolve_identifier(identifier)\n            if metadata["source_url"] not in seen_urls:\n                resolved.append((identifier, metadata))\n                seen_urls.add(metadata["source_url"])\n        if max_structures is not None and len(resolved) + len(local_entries) > max_structures:\n            raise ValueError("Input exceeds the configured structure-count guard.")\n        if not resolved and not local_entries:\n            raise ValueError("Provide at least one supported identifier or structure.")\n\n        def add(name, data, metadata):\n            nonlocal total_bytes\n            data = structure_bytes(data, name, max_file_bytes)\n            total_bytes += len(data)\n            if max_total_bytes is not None and total_bytes > max_total_bytes:\n                raise ValueError("Combined structure data exceeds the configured total-byte guard.")\n            fmt, original = structure_format(name), PurePosixPath(name).name\n            stem = re.sub(r"[^A-Za-z0-9_.-]+", "_", original[:-3] if original.lower().endswith(".gz") else original)\n            stem = Path(stem).stem.strip("._-")[:100] or "structure"\n            archive_path = f"structures/{len(entries) + 1:04d}_{stem}.{fmt}"\n            entry = {"path": archive_path, "original_name": original, "file_format": fmt,\n                     "size_bytes": len(data), "sha256": hashlib.sha256(data).hexdigest(), **metadata}\n            if name != original:\n                entry["archive_member"] = name\n            blob = work / f"structure-{len(entries)}"\n            blob.write_bytes(data)\n            entries.append(entry)\n            blobs[archive_path] = blob\n\n        for identifier, metadata in resolved:\n            add(metadata["filename"], downloader(metadata["source_url"], max_bytes=max_file_bytes),\n                {"source_type": "identifier", "input_id": identifier,\n                 "canonical_id": metadata["canonical_id"], "source_url": metadata["source_url"]})\n        for name, raw_path, metadata in local_entries:\n            add(name, raw_path.read_bytes(), metadata)\n            raw_path.unlink()\n\n        # Use measured compressed structure sizes plus a conservative metadata\n        # allowance to form upload parts, then verify the actual final ZIP size.\n        groups, current, estimate = [], [], 1024\n        for entry in entries:\n            member_cost = (_compressed_size(blobs[entry["path"]]) + 76 + 2 * len(entry["path"].encode())\n                           + len(_json_bytes(entry)) + 256)\n            if current and estimate + member_cost > upload_max_bytes:\n                groups.append(current)\n                current, estimate = [], 1024\n            current.append(entry)\n            estimate += member_cost\n        if current:\n            groups.append(current)\n\n        while True:\n            archives, manifests, split = [], [], None\n            for index, group in enumerate(groups, 1):\n                name = output.name if len(groups) == 1 else f"{output.stem}.part{index:03d}.zip"\n                for entry in group:\n                    entry["archive_file"] = name\n                archive_record = {"path": name, "structure_count": len(group),\n                                  "total_structure_bytes": sum(entry["size_bytes"] for entry in group)}\n                part_ids = list(dict.fromkeys(entry["input_id"] for entry in group if "input_id" in entry))\n                manifest = _manifest(group, identifiers if len(groups) == 1 else part_ids)\n                manifest["archives"] = [archive_record]\n                candidate = work / f"part-{index}.zip"\n                _write_archive(candidate, manifest, blobs)\n                if candidate.stat().st_size > upload_max_bytes:\n                    if len(group) == 1:\n                        raise ValueError("A single compressed structure cannot fit the configured Tamarind upload-byte limit; no inputs were omitted.")\n                    split = index - 1\n                    break\n                archives.append((output.parent / name, candidate))\n                manifests.append(archive_record)\n            if split is None:\n                break\n            group = groups[split]\n            groups[split:split + 1] = [group[:len(group) // 2], group[len(group) // 2:]]\n\n        manifest = _manifest(entries, identifiers)\n        manifest["archives"] = manifests\n        staged_manifest = work / "manifest.json"\n        staged_manifest.write_bytes(_json_bytes(manifest))\n        publications = [*archives, (manifest_path, staged_manifest)]\n        if not overwrite and any(destination.exists() for destination, _ in publications):\n            raise FileExistsError("An output already exists; choose another path or use --force.")\n        # Stage on the output filesystem, retaining previous files for rollback.\n        with tempfile.TemporaryDirectory(prefix=".zincsight-", dir=output.parent) as transaction:\n            staging, backups, published = Path(transaction), [], []\n            for index, (_, original) in enumerate(publications):\n                with original.open("rb") as source_stream, (staging / str(index)).open("wb") as target:\n                    for block in iter(lambda: source_stream.read(1024 * 1024), b""):\n                        target.write(block)\n            try:\n                destinations = {destination for destination, _ in publications}\n                obsolete = [candidate for candidate in output.parent.iterdir()\n                            if candidate.is_file() and candidate not in destinations\n                            and (candidate == output or re.fullmatch(\n                                re.escape(output.stem) + r"\\.part[0-9]+\\.zip", candidate.name))]\n                if overwrite:\n                    for index, destination in enumerate(obsolete):\n                        backup = staging / f"obsolete-{index}"\n                        os.replace(destination, backup)\n                        backups.append((destination, backup))\n                for index, (destination, _) in enumerate(publications):\n                    if destination.exists():\n                        backup = staging / f"backup-{index}"\n                        os.replace(destination, backup)\n                        backups.append((destination, backup))\n                    os.replace(staging / str(index), destination)\n                    published.append(destination)\n            except BaseException:\n                for destination in reversed(published):\n                    destination.unlink(missing_ok=True)\n                for destination, backup in reversed(backups):\n                    os.replace(backup, destination)\n                raise\n    return manifest\n\n\ndef main(argv=None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("sources", nargs="*", help="Structure files, folders, archives, or ID-list files")\n    parser.add_argument("--ids", default="", help="Comma/whitespace-separated public structure identifiers")\n    parser.add_argument("--ids-file", action="append", default=[], help="UTF-8 .txt/.csv/.tsv ID list; may be repeated")\n    parser.add_argument("--output", required=True, type=Path, help="ZIP basename; large batches produce numbered upload parts")\n    parser.add_argument("--force", action="store_true", help="Replace an existing prepared output family")\n    for option, help_text in (("max-structures", "structure count"), ("max-file-bytes", "bytes per structure"),\n                              ("max-total-bytes", "total uncompressed structure bytes"), ("max-archive-entries", "archive entries"),\n                              ("max-folder-entries", "folder entries"), ("max-id-file-bytes", "ID-list bytes")):\n        parser.add_argument("--" + option, type=int, default=None, help="Optional guard: " + help_text + "; disabled by default")\n    parser.add_argument("--upload-max-bytes", type=int, default=UPLOAD_MAX_BYTES,\n                        help="Maximum bytes per prepared ZIP (default: Tamarind\'s 200 MB upload limit)")\n    args = parser.parse_args(argv)\n    try:\n        manifest = prepare_inputs(args.output, ids=args.ids, id_files=args.ids_file, sources=args.sources,\n                                  overwrite=args.force, max_structures=args.max_structures,\n                                  max_file_bytes=args.max_file_bytes, max_total_bytes=args.max_total_bytes,\n                                  max_archive_entries=args.max_archive_entries, max_folder_entries=args.max_folder_entries,\n                                  max_id_file_bytes=args.max_id_file_bytes, upload_max_bytes=args.upload_max_bytes)\n    except (ValueError, OSError, UnicodeError, zipfile.BadZipFile, tarfile.TarError) as exc:\n        parser.exit(2, f"Preparation failed: {exc}\\n")\n    print(f"Prepared all {manifest[\'structure_count\']} structures in {len(manifest[\'archives\'])} upload archive(s):")\n    for archive in manifest["archives"]:\n        print(args.output.parent / archive["path"])\n    print(f"Manifest: {args.output.with_suffix(\'.manifest.json\')}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
PREPARER_SHA256 = '47fe46cec2570b06a487037ac7fdf7ac978f447dfc4e99d350ef8be1cdd79c41'
assert hashlib.sha256(PREPARER_SOURCE.encode()).hexdigest() == PREPARER_SHA256
preparer = types.ModuleType("zincsight_tamarind_preparer")
preparer.__file__ = "embedded_prepare_inputs.py"
exec(compile(PREPARER_SOURCE, preparer.__file__, "exec"), preparer.__dict__)

ids_form = widgets.Textarea(
    value="", placeholder="1KLS, P69905\nAF-A0A3W0S055-F1-model_v6.cif",
    description="IDs:", rows=6, layout=widgets.Layout(width="100%"))
id_files_form = widgets.Checkbox(value=False, description="Upload ID-list files (.txt/.csv/.tsv)", indent=False)
structures_form = widgets.Checkbox(value=False, description="Upload structures or archives", indent=False)
display(widgets.VBox([
    widgets.HTML("<b>Paste IDs separated by commas, spaces, tabs, or newlines.</b> Leave blank when uploading files only."),
    ids_form, id_files_form, structures_form,
    widgets.HTML("Edit these inputs, then run <b>2. Prepare ZIP</b> below. No prediction job is submitted.")]))


In [ ]:
#@title 2. Prepare ZIP { display-mode: "form" }
from google.colab import files
import tempfile

work = Path(tempfile.mkdtemp(prefix="zincsight-inputs-", dir="/content"))

def upload_inputs(subfolder, prompt):
    print(prompt)
    uploaded = files.upload(target_dir=str(work / subfolder))
    return [Path(filename) for filename in uploaded]

id_paths = upload_inputs("id_lists", "Choose one or more ID-list files:") if id_files_form.value else []
structure_paths = upload_inputs("structures", "Choose structures and/or archives:") if structures_form.value else []
ids_text = ids_form.value
if not ids_text.strip() and not id_paths and not structure_paths:
    raise ValueError("Enter at least one ID or select an upload option above, then rerun this cell.")
prepared_zip = work / "zincsight_inputs.zip"
manifest = preparer.prepare_inputs(prepared_zip, ids=ids_text, id_files=id_paths, sources=structure_paths)
prepared_paths = [work / archive["path"] for archive in manifest["archives"]]
print(f"Ready: all {manifest['structure_count']} structures in {len(prepared_paths)} upload archive(s).")
for archive in prepared_paths:
    print(f"{archive.name}: {archive.stat().st_size:,} bytes")
print("Run Download below, then upload every generated ZIP part together in Tamarind.")


In [ ]:
#@title 3. Download { display-mode: "form" }
from google.colab import files

if "prepared_paths" not in globals() or not prepared_paths or not all(path.is_file() for path in prepared_paths):
    raise ValueError("Run Prepare ZIP successfully before downloading.")
print("Allow multiple browser downloads if prompted. Upload all these ZIP parts together in Tamarind.")
for path in prepared_paths:
    files.download(str(path))
print("Open ZincSight in Tamarind, select every ZIP under Protein structures or batch archives, choose prediction settings, and run.")
print("Each part already includes its provenance manifest; no separate manifest upload is needed.")
